In [1]:
# ============================================================================
# BLOCK 1: ENHANCED IMPORTS AND DEPENDENCIES
# ============================================================================

# Install required packages (run this first in Colab)
!pip install rdkit pandas numpy scikit-learn matplotlib seaborn imbalanced-learn xgboost optuna tensorflow

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem.Fingerprints import FingerprintMols
from rdkit.Chem import MACCSkeys
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator, GetAtomPairGenerator

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import optuna
import pickle
import warnings
warnings.filterwarnings('ignore')

"""# Connect to Drive (for Colab)
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ToxPred"""

"# Connect to Drive (for Colab)\nfrom google.colab import drive\ndrive.mount('/content/drive')\n%cd /content/drive/MyDrive/ToxPred"

In [2]:
# ============================================================================
# BLOCK 2: ENHANCED TOXICITY PREDICTOR CLASS WITH PRE-OPTIMIZED PARAMETERS
# ============================================================================

import lightgbm as lgb # Added LightGBM import

class EnhancedToxicityPredictor:
    def __init__(self):
        self.scaler = StandardScaler()
        self.feature_selector = SelectKBest(f_classif, k='all') # Changed k to 'all'
        self.models = {}
        self.ensemble_model = None
        self.best_params = {}
        self.feature_names = None
        self.is_trained = False
        self.nn_model = None

        # Pre-optimized parameters from Trial 26 (score: 0.9072085889570553) - Need to add LightGBM params here if optimizing
        self.pre_optimized_params = {
            'rf_n_estimators': 236,
            'rf_max_depth': 23,
            'rf_min_samples_split': 18,
            'rf_min_samples_leaf': 8,
            'xgb_n_estimators': 140,
            'xgb_max_depth': 9,
            'xgb_learning_rate': 0.2507830493666598,
            'xgb_subsample': 0.6094213865403939,
            'xgb_colsample_bytree': 0.7071271107568533,
            'svm_C': 59.03505324844556,
            'svm_gamma': 0.593644676772998,
            'svm_kernel': 'rbf',
            # Added placeholder LightGBM parameters (adjust during optimization)
            'lgb_n_estimators': 100,
            'lgb_num_leaves': 31,
            'lgb_learning_rate': 0.1
        }


    def validate_smiles(self, df):
        """Validate SMILES strings efficiently"""
        valid_mask = df['SMILES'].apply(lambda x: Chem.MolFromSmiles(x) is not None)
        invalid_count = (~valid_mask).sum()
        print(f"Invalid SMILES found: {invalid_count}")
        return df[valid_mask].reset_index(drop=True)

    def get_molecular_fingerprints(self, mol):
        """Get various molecular fingerprints using updated RDKit generators"""
        fingerprints = {}

        try:
            # Morgan fingerprints (ECFP4)
            morgan_gen = GetMorganGenerator(radius=2, fpSize=1024)
            morgan_fp = morgan_gen.GetFingerprint(mol)
            fingerprints.update({f'Morgan_{i}': int(morgan_fp[i]) for i in range(1024)})

            # MACCS keys
            maccs_fp = MACCSkeys.GenMACCSKeys(mol)
            fingerprints.update({f'MACCS_{i}': int(maccs_fp[i]) for i in range(len(maccs_fp))})

            # Atom pairs
            atom_pair_gen = GetAtomPairGenerator(maxDistance=30, fpSize=1024)
            atom_pair_fp = atom_pair_gen.GetFingerprint(mol)
            fingerprints.update({f'AtomPair_{i}': int(atom_pair_fp[i]) for i in range(1024)})

            # Topological fingerprints (RDKit)
            rdkit_fp = FingerprintMols.FingerprintMol(mol)
            rdkit_bits = rdkit_fp.GetOnBits()
            for i, bit in enumerate(rdkit_bits):
                if i < 512:
                    fingerprints[f'RDKit_{bit}'] = 1

        except Exception as e:
            print(f"Error generating fingerprints: {e}")

        return fingerprints

    def get_toxicity_descriptors(self, mol):
        """Get descriptors specifically relevant to toxicity prediction"""
        descriptors = {}

        try:
            # Basic physicochemical properties
            descriptors.update({
                'MW': Descriptors.MolWt(mol),
                'LogP': Descriptors.MolLogP(mol),
                'TPSA': Descriptors.TPSA(mol),
                'HBD': Descriptors.NumHDonors(mol),
                'HBA': Descriptors.NumHAcceptors(mol),
                'RotBonds': Descriptors.NumRotatableBonds(mol),
                'AromaticRings': Descriptors.NumAromaticRings(mol),
                'FractionCSP3': Descriptors.FractionCSP3(mol),
                'MolMR': Descriptors.MolMR(mol),
                'HeavyAtomCount': Descriptors.HeavyAtomCount(mol),
                'RingCount': Descriptors.RingCount(mol),
                'NumHeteroatoms': Descriptors.NumHeteroatoms(mol),
                'NumAliphaticRings': Descriptors.NumAliphaticRings(mol),
                'NumSaturatedRings': Descriptors.NumSaturatedRings(mol),
                'NumAromaticCarbocycles': Descriptors.NumAromaticCarbocycles(mol),
                'LabuteASA': Descriptors.LabuteASA(mol),
                'BalabanJ': Descriptors.BalabanJ(mol),
                'Chi0': Descriptors.Chi0(mol),
                'Chi1': Descriptors.Chi1(mol),
                'Chi0v': Descriptors.Chi0v(mol),
                'Chi1v': Descriptors.Chi1v(mol),
                'Kappa1': Descriptors.Kappa1(mol),
                'Kappa2': Descriptors.Kappa2(mol),
                'Kappa3': Descriptors.Kappa3(mol),
                'MaxEStateIndex': Descriptors.MaxEStateIndex(mol),
                'MinEStateIndex': Descriptors.MinEStateIndex(mol),
                'SlogP_VSA2': Descriptors.SlogP_VSA2(mol),
                'SMR_VSA3': Descriptors.SMR_VSA3(mol),
                'PEOE_VSA6': Descriptors.PEOE_VSA6(mol),
                'EState_VSA2': Descriptors.EState_VSA2(mol),
            })

            # Toxicity-specific descriptors
            descriptors.update({
                'NumRadicalElectrons': Descriptors.NumRadicalElectrons(mol),
                'NumValenceElectrons': Descriptors.NumValenceElectrons(mol),
                'NumAromaticHeterocycles': Descriptors.NumAromaticHeterocycles(mol),
                'NumAliphaticHeterocycles': Descriptors.NumAliphaticHeterocycles(mol),
                'NumSaturatedHeterocycles': Descriptors.NumSaturatedHeterocycles(mol),
                'fr_Al_COO': Descriptors.fr_Al_COO(mol),
                'fr_Al_OH': Descriptors.fr_Al_OH(mol),
                'fr_Ar_N': Descriptors.fr_Ar_N(mol),
                'fr_Ar_NH': Descriptors.fr_Ar_NH(mol),
                'fr_Ar_OH': Descriptors.fr_Ar_OH(mol),
                'fr_COO': Descriptors.fr_COO(mol),
                'fr_COO2': Descriptors.fr_COO2(mol),
                'fr_C_O': Descriptors.fr_C_O(mol),
                'fr_C_S': Descriptors.fr_C_S(mol),
                'fr_HOCCN': Descriptors.fr_HOCCN(mol),
                'fr_Imine': Descriptors.fr_Imine(mol),
                'fr_NH0': Descriptors.fr_NH0(mol),
                'fr_NH1': Descriptors.fr_NH1(mol),
                'fr_NH2': Descriptors.fr_NH2(mol),
                'fr_N_O': Descriptors.fr_N_O(mol),
                'fr_Ndealkylation1': Descriptors.fr_Ndealkylation1(mol),
                'fr_Ndealkylation2': Descriptors.fr_Ndealkylation2(mol),
                'fr_Nhpyrrole': Descriptors.fr_Nhpyrrole(mol),
                'fr_SH': Descriptors.fr_SH(mol),
                'fr_aldehyde': Descriptors.fr_aldehyde(mol),
                'fr_benzene': Descriptors.fr_benzene(mol),
                'fr_furan': Descriptors.fr_furan(mol),
                'fr_halogen': Descriptors.fr_halogen(mol),
                'fr_ketone': Descriptors.fr_ketone(mol),
                'fr_lactam': Descriptors.fr_lactam(mol),
                'fr_lactone': Descriptors.fr_lactone(mol),
                'fr_methoxy': Descriptors.fr_methoxy(mol),
                'fr_nitro': Descriptors.fr_nitro(mol),
                'fr_oxazole': Descriptors.fr_oxazole(mol),
                'fr_phenol': Descriptors.fr_phenol(mol),
                'fr_pyridine': Descriptors.fr_pyridine(mol),
                'fr_quatN': Descriptors.fr_quatN(mol),
                'fr_sulfide': Descriptors.fr_sulfide(mol),
                'fr_sulfonamd': Descriptors.fr_sulfonamd(mol),
                'fr_thiazole': Descriptors.fr_thiazole(mol),
                'fr_thiophene': Descriptors.fr_thiophene(mol),
                'fr_unbrch_alkane': Descriptors.fr_unbrch_alkane(mol),
                'fr_urea': Descriptors.fr_urea(mol),
            })

            # Handle NaN/Inf values
            for key, value in descriptors.items():
                if np.isnan(value) or np.isinf(value):
                    descriptors[key] = 0.0

        except Exception as e:
            print(f"Error calculating descriptors: {e}")

        return descriptors

    def compute_features(self, df):
        """Compute all features efficiently"""
        print("Computing molecular features...")

        features_list = []
        failed_count = 0

        for idx, row in df.iterrows():
            if idx % 1000 == 0:
                print(f"Processed {idx}/{len(df)} molecules")

            smiles = row['SMILES']
            mmp = row['MMP']

            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                failed_count += 1
                continue

            # Get descriptors and fingerprints
            descriptors = self.get_toxicity_descriptors(mol)
            fingerprints = self.get_molecular_fingerprints(mol)

            # Combine all features
            features = {'SMILES': smiles, 'MMP': mmp}
            features.update(descriptors)
            features.update(fingerprints)

            features_list.append(features)

        print(f"Failed to process {failed_count} molecules")

        # Create DataFrame
        df_features = pd.DataFrame(features_list)

        # Separate SMILES and MMP before filtering features
        df_smiles_mmp = df_features[['SMILES', 'MMP']].copy()
        df_features = df_features.drop(columns=['SMILES', 'MMP'])

        # Remove features with zero variance
        feature_cols = df_features.columns
        cols_to_keep = df_features.nunique() > 1
        df_features = df_features.loc[:, cols_to_keep]

        # Concatenate the filtered features with SMILES and MMP
        df_features = pd.concat([df_features, df_smiles_mmp], axis=1)

        print(f"Feature computation completed. Shape: {df_features.shape}")
        return df_features

    def analyze_toxicity_components(self, X, y, feature_names):
        """Analyze which components are responsible for toxicity"""
        print("\n" + "="*50)
        print("TOXICITY COMPONENT ANALYSIS")
        print("="*50)

        # Feature importance from Random Forest
        rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_temp.fit(X, y)

        feature_importance = pd.DataFrame({
            'feature': feature_names,
            'importance': rf_temp.feature_importances_
        }).sort_values('importance', ascending=False)

        print("\nTop 20 Most Important Features for Toxicity:")
        print(feature_importance.head(20).to_string(index=False))

        # Analyze by feature type
        feature_types = {
            'Descriptors': [f for f in feature_names if not any(fp in f for fp in ['Morgan', 'MACCS', 'AtomPair', 'RDKit'])],
            'Morgan_Fingerprints': [f for f in feature_names if 'Morgan' in f],
            'MACCS_Keys': [f for f in feature_names if 'MACCS' in f],
            'Atom_Pairs': [f for f in feature_names if 'AtomPair' in f],
            'RDKit_Fingerprints': [f for f in feature_names if 'RDKit' in f]
        }

        print("\nFeature Type Analysis:")
        for ftype, features in feature_types.items():
            if features:
                type_importance = feature_importance[feature_importance['feature'].isin(features)]
                avg_importance = type_importance['importance'].mean()
                top_feature = type_importance.iloc[0] if len(type_importance) > 0 else None
                print(f"{ftype}: {len(features)} features, avg importance: {avg_importance:.4f}")
                if top_feature is not None:
                    print(f"  Top feature: {top_feature['feature']} (importance: {top_feature['importance']:.4f})")

        # Identify toxic substructures
        print("\nIdentifying Toxic Substructures:")
        toxic_descriptors = feature_importance[feature_importance['feature'].str.contains('fr_')].head(10)
        print(toxic_descriptors.to_string(index=False))

        return feature_importance


    def create_neural_network(self, input_dim, trial=None):
        """Create neural network model with optional Optuna optimization"""
        if trial:
            # Optuna hyperparameter optimization
            n_layers = trial.suggest_int('n_layers', 2, 4)
            dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
            learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

            model = Sequential()
            model.add(Dense(trial.suggest_int('units_0', 64, 512),
                          activation='relu', input_shape=(input_dim,)))
            model.add(BatchNormalization())
            model.add(Dropout(dropout_rate))

            for i in range(n_layers - 1):
                model.add(Dense(trial.suggest_int(f'units_{i+1}', 32, 256),
                              activation='relu'))
                model.add(BatchNormalization())
                model.add(Dropout(dropout_rate))

            model.add(Dense(1, activation='sigmoid'))
            model.compile(optimizer=Adam(learning_rate=learning_rate),
                         loss='binary_crossentropy', metrics=['accuracy'])
        else:
            # Default architecture
            model = Sequential([
                Dense(256, activation='relu', input_shape=(input_dim,)),
                BatchNormalization(),
                Dropout(0.3),
                Dense(128, activation='relu'),
                BatchNormalization(),
                Dropout(0.3),
                Dense(64, activation='relu'),
                Dropout(0.2),
                Dense(1, activation='sigmoid')
            ])
            model.compile(optimizer=Adam(learning_rate=0.001),
                         loss='binary_crossentropy', metrics=['accuracy'])

        return model

    def use_pre_optimized_params(self):
        """Use the pre-optimized parameters from Trial 26"""
        print("🚀 Using pre-optimized parameters from Trial 26")
        print(f"   Expected accuracy: ~90.72% (Trial 26 score: 0.9072085889570553)")
        print(f"   Skipping hyperparameter optimization to save computation time...")

        self.best_params = self.pre_optimized_params.copy()
        print(f"   Parameters loaded: {len(self.best_params)} hyperparameters set")
        return self.best_params

    def optimize_hyperparameters(self, X_train, y_train, X_val, y_val, use_pre_optimized=True):
        """Optimize hyperparameters using Optuna with option to use pre-optimized params"""

        if use_pre_optimized:
            print("="*60)
            print("USING PRE-OPTIMIZED PARAMETERS")
            print("="*60)
            return self.use_pre_optimized_params()

        # Original optimization code (only runs if use_pre_optimized=False)
        print("Optimizing hyperparameters with Optuna...")
        print("Target accuracy: >90.5% - Will stop early if reached")

        def objective(trial):
            # Random Forest parameters
            rf_params = {
                'n_estimators': trial.suggest_int('rf_n_estimators', 50, 300),
                'max_depth': trial.suggest_int('rf_max_depth', 5, 30),
                'min_samples_split': trial.suggest_int('rf_min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('rf_min_samples_leaf', 1, 10),
            }

            # XGBoost parameters
            xgb_params = {
                'n_estimators': trial.suggest_int('xgb_n_estimators', 50, 300),
                'max_depth': trial.suggest_int('xgb_max_depth', 3, 10),
                'learning_rate': trial.suggest_float('xgb_learning_rate', 0.01, 0.3),
                'subsample': trial.suggest_float('xgb_subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('xgb_colsample_bytree', 0.6, 1.0),
            }

            # SVM parameters
            svm_params = {
                'C': trial.suggest_float('svm_C', 0.1, 100, log=True),
                'gamma': trial.suggest_float('svm_gamma', 1e-4, 1, log=True),
                'kernel': trial.suggest_categorical('svm_kernel', ['rbf', 'linear', 'poly'])
            }

            # LightGBM parameters
            lgb_params = {
                'objective': 'binary',
                'metric': 'binary_logloss',
                'n_estimators': trial.suggest_int('lgb_n_estimators', 50, 300),
                'num_leaves': trial.suggest_int('lgb_num_leaves', 2, 64),
                'learning_rate': trial.suggest_float('lgb_learning_rate', 0.01, 0.3),
                'feature_fraction': trial.suggest_float('lgb_feature_fraction', 0.6, 1.0),
                'bagging_fraction': trial.suggest_float('lgb_bagging_fraction', 0.6, 1.0),
                'bagging_freq': trial.suggest_int('lgb_bagging_freq', 0, 10),
            }


            # Train models with suggested parameters
            rf = RandomForestClassifier(**rf_params, random_state=42, class_weight='balanced')
            xgb_model = xgb.XGBClassifier(**xgb_params, random_state=42, eval_metric='logloss')
            svm_model = SVC(**svm_params, random_state=42, class_weight='balanced', probability=True)
            lgb_model = lgb.LGBMClassifier(**lgb_params, random_state=42, class_weight='balanced') # Added LightGBM

            # Ensemble voting
            ensemble = VotingClassifier(
                estimators=[('rf', rf), ('xgb', xgb_model), ('svm', svm_model), ('lgb', lgb_model)], # Added LightGBM
                voting='soft'
            )

            ensemble.fit(X_train, y_train)
            y_pred = ensemble.predict(X_val)
            accuracy = accuracy_score(y_val, y_pred)

            # Print progress
            print(f"Trial {trial.number}: Accuracy = {accuracy:.4f}")

            # Early stopping condition: if accuracy > 0.905, stop the optimization
            if accuracy > 0.905:
                print(f"🎯 Target accuracy reached! Accuracy: {accuracy:.4f} > 0.905")
                # Prune the study to stop further trials
                trial.study.stop()

            return accuracy

        # Create study with early stopping callback
        study = optuna.create_study(direction='maximize')

        try:
            # Run optimization with maximum 50 trials, but may stop early
            study.optimize(objective, n_trials=50)
        except KeyboardInterrupt:
            print("Optimization stopped by user")
        except Exception as e:
            print(f"Optimization completed or stopped: {e}")

        # Check if we reached our target
        if study.best_value > 0.905:
            print(f"🎉 SUCCESS! Best accuracy: {study.best_value:.4f} > 0.905")
            print(f"Total trials completed: {len(study.trials)}")
        else:
            print(f"Best accuracy achieved: {study.best_value:.4f}")
            print(f"Total trials completed: {len(study.trials)}")

        self.best_params = study.best_params
        print(f"Best parameters found: {self.best_params}")
        return study.best_params

    def train_multiple_models(self, X_train, y_train, X_val, y_val, optimize=True, use_pre_optimized=True):
        """Train multiple models with optional hyperparameter optimization"""
        print("Training multiple models...")

        if optimize:
            best_params = self.optimize_hyperparameters(X_train, y_train, X_val, y_val, use_pre_optimized=use_pre_optimized)

            # Extract parameters for each model
            rf_params = {k.replace('rf_', ''): v for k, v in best_params.items() if k.startswith('rf_')}
            xgb_params = {k.replace('xgb_', ''): v for k, v in best_params.items() if k.startswith('xgb_')}
            svm_params = {k.replace('svm_', ''): v for k, v in best_params.items() if k.startswith('svm_')}
            lgb_params = {k.replace('lgb_', ''): v for k, v in best_params.items() if k.startswith('lgb_')} # Added LightGBM params
        else:
            # Default parameters
            rf_params = {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2}
            xgb_params = {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1}
            svm_params = {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf'}
            lgb_params = {'n_estimators': 100, 'num_leaves': 31, 'learning_rate': 0.1} # Added LightGBM default params


        # Add fixed random_state for reproducibility
        rf_params['random_state'] = 42
        xgb_params['random_state'] = 42
        svm_params['random_state'] = 42
        lgb_params['random_state'] = 42 # Added LightGBM random state

        print(f"🔧 Using parameters:")
        print(f"   Random Forest: {rf_params}")
        print(f"   XGBoost: {xgb_params}")
        print(f"   SVM: {svm_params}")
        print(f"   LightGBM: {lgb_params}") # Added LightGBM params printout

        # Train individual models
        print("Training Random Forest...")
        self.models['rf'] = RandomForestClassifier(**rf_params, class_weight='balanced')
        self.models['rf'].fit(X_train, y_train)

        print("Training XGBoost...")
        self.models['xgb'] = xgb.XGBClassifier(**xgb_params, eval_metric='logloss')
        self.models['xgb'].fit(X_train, y_train)

        print("Training SVM...")
        self.models['svm'] = SVC(**svm_params, class_weight='balanced', probability=True)
        self.models['svm'].fit(X_train, y_train)

        print("Training LightGBM...") # Added LightGBM training
        self.models['lgb'] = lgb.LGBMClassifier(**lgb_params, class_weight='balanced')
        self.models['lgb'].fit(X_train, y_train)

        print("Training Neural Network...")
        self.nn_model = self.create_neural_network(X_train.shape[1])
        early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        self.nn_model.fit(X_train, y_train,
                         validation_data=(X_val, y_val),
                         epochs=100, batch_size=32,
                         callbacks=[early_stopping], verbose=0)

        # Create ensemble model
        print("Creating ensemble model...")
        self.ensemble_model = VotingClassifier(
            estimators=[('rf', self.models['rf']),
                       ('xgb', self.models['xgb']),
                       ('svm', self.models['svm']),
                       ('lgb', self.models['lgb'])], # Added LightGBM to ensemble
            voting='soft'
        )
        self.ensemble_model.fit(X_train, y_train)

        self.is_trained = True
        print("✅ All models trained successfully!")
        return self.models

    def evaluate_models(self, X_test, y_test):
        """Evaluate all trained models"""
        print("\n" + "="*50)
        print("MODEL EVALUATION")
        print("="*50)

        results = {}

        for name, model in self.models.items():
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)[:, 1]

            accuracy = accuracy_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_pred_proba)

            results[name] = {'accuracy': accuracy, 'auc': auc}
            print(f"{name.upper()} - Accuracy: {accuracy:.4f}, AUC: {auc:.4f}")

        # Neural Network evaluation
        nn_pred_proba = self.nn_model.predict(X_test).flatten()
        nn_pred = (nn_pred_proba > 0.5).astype(int)
        nn_accuracy = accuracy_score(y_test, nn_pred)
        nn_auc = roc_auc_score(y_test, nn_pred_proba)

        results['nn'] = {'accuracy': nn_accuracy, 'auc': nn_auc}
        print(f"NEURAL NETWORK - Accuracy: {nn_accuracy:.4f}, AUC: {nn_auc:.4f}")

        # Ensemble evaluation
        ensemble_pred = self.ensemble_model.predict(X_test)
        ensemble_pred_proba = self.ensemble_model.predict_proba(X_test)[:, 1]
        ensemble_accuracy = accuracy_score(y_test, ensemble_pred)
        ensemble_auc = roc_auc_score(y_test, ensemble_pred_proba)

        results['ensemble'] = {'accuracy': ensemble_accuracy, 'auc': ensemble_auc}
        print(f"ENSEMBLE - Accuracy: {ensemble_accuracy:.4f}, AUC: {ensemble_auc:.4f}")

        return results

    def test_known_disruptors(self, known_disruptors):
        """Test the model on known MMP disruptors"""
        print("\n" + "="*50)
        print("TESTING KNOWN MMP DISRUPTORS")
        print("="*50)

        correct_predictions = 0
        total_predictions = 0

        for smiles in known_disruptors:
            try:
                predictions, error = self.predict_single_molecule_enhanced(smiles)
                if error:
                    continue

                # Use ensemble prediction
                ensemble_pred = predictions['ensemble']['prediction']
                ensemble_prob = predictions['ensemble']['probability']

                print(f"SMILES: {smiles}")
                print(f"Ensemble Prediction: {'Toxic' if ensemble_pred == 1 else 'Non-toxic'} (probability: {ensemble_prob:.3f})")
                print("-" * 40)

                if ensemble_pred == 1:
                    correct_predictions += 1
                total_predictions += 1

            except Exception as e:
                print(f"Error processing {smiles}: {e}")

        accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
        print(f"\nKnown MMP disruptors correctly identified: {correct_predictions}/{total_predictions} ({accuracy:.2%})")

        return accuracy

    def predict_single_molecule_enhanced(self, smiles):
        """Enhanced prediction using all models"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None, "Invalid SMILES string"

            # Get features
            descriptors = self.get_toxicity_descriptors(mol)
            fingerprints = self.get_molecular_fingerprints(mol)
            features = {}
            features.update(descriptors)
            features.update(fingerprints)

            # Create feature vector
            feature_vector = np.array([features.get(name, 0) for name in self.feature_names])
            feature_vector = feature_vector.reshape(1, -1)

            # Scale and select features
            feature_vector_scaled = self.scaler.transform(feature_vector)
            feature_vector_selected = self.feature_selector.transform(feature_vector_scaled)

            # Get predictions from all models
            predictions = {}

            for name, model in self.models.items():
                if name == 'nn': # NN prediction is handled separately due to Keras model
                    continue
                pred = model.predict(feature_vector_selected)[0]
                proba = model.predict_proba(feature_vector_selected)[0]
                predictions[name] = {'prediction': pred, 'probability': proba[1]}

            # Neural network prediction
            nn_proba = self.nn_model.predict(feature_vector_selected)[0][0]
            nn_pred = 1 if nn_proba > 0.5 else 0
            predictions['nn'] = {'prediction': nn_pred, 'probability': nn_proba}


            # Ensemble prediction
            ensemble_pred = self.ensemble_model.predict(feature_vector_selected)[0]
            ensemble_proba = self.ensemble_model.predict_proba(feature_vector_selected)[0]
            predictions['ensemble'] = {'prediction': ensemble_pred, 'probability': ensemble_proba[1]}

            return predictions, None

        except Exception as e:
            return None, f"Error processing molecule: {str(e)}"

    def save_enhanced_model(self, model_path="Talksick-v2-ensemble_toxicity_model.pkl"):
        """Save all trained models"""
        if not self.is_trained:
            print("No trained models to save")
            return False

        try:
            model_data = {
                'models': {name: model for name, model in self.models.items() if name != 'nn'}, # Exclude NN
                'ensemble_model': self.ensemble_model,
                'scaler': self.scaler,
                'feature_selector': self.feature_selector,
                'feature_names': self.feature_names,
                'best_params': self.best_params,
                'pre_optimized_params': self.pre_optimized_params  # Save the pre-optimized params too
            }

            with open(model_path, 'wb') as f:
                pickle.dump(model_data, f)

            # Save neural network separately
            if self.nn_model:
                self.nn_model.save(model_path.replace('.pkl', '_nn.h5'))

            print(f"✅ Enhanced models saved successfully to {model_path}")
            return True

        except Exception as e:
            print(f"❌ Error saving models: {e}")
            return False

    def load_enhanced_model(self, model_path="Talksick-v2-ensemble_toxicity_model.pkl"):
        """Load all saved models"""
        try:
            with open(model_path, 'rb') as f:
                model_data = pickle.load(f)

            self.models = model_data['models']
            self.ensemble_model = model_data['ensemble_model']
            self.scaler = model_data['scaler']
            self.feature_selector = model_data['feature_selector']
            self.feature_names = model_data['feature_names']
            self.best_params = model_data['best_params']
            self.is_trained = True

            # Load neural network
            from tensorflow.keras.models import load_model
            nn_model_path = model_path.replace('.pkl', '_nn.h5') # Corrected filename
            self.nn_model = load_model(nn_model_path)
            self.models['nn'] = self.nn_model # Add NN back to models dict after loading


            print(f"✅ Enhanced models loaded successfully from {model_path}")
            return True

        except Exception as e:
            print(f"❌ Error loading models: {e}")
            return False

In [3]:
# ============================================================================
# BLOCK 3: ENHANCED TRAINING FUNCTION
# ============================================================================

def train_enhanced_toxicity_model(input_file="sr-mmp-monol.smiles", optimize_hyperparams=True):
    """Enhanced training pipeline with multiple algorithms"""
    print("="*60)
    print("ENHANCED TOXICITY PREDICTION MODEL TRAINING")
    print("="*60)

    # Initialize enhanced predictor
    predictor = EnhancedToxicityPredictor()

    # Load and prepare data (same as before)
    print(f"Loading data from {input_file}...")
    try:
        df = pd.read_csv(input_file, sep="\s+", header=None, usecols=[0, 2], names=["SMILES", "MMP"])
    except FileNotFoundError:
        print(f"❌ Error: Input file '{input_file}' not found.")
        return None, None


    print(f"Original dataset shape: {df.shape}")
    df = predictor.validate_smiles(df)
    df_features = predictor.compute_features(df)

    # Prepare features
    X = df_features.drop(columns=['MMP', 'SMILES'])
    y = df_features['MMP']
    predictor.feature_names = X.columns.tolist()

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Further split training for validation
    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )

    # Feature scaling and selection
    X_train_scaled = predictor.scaler.fit_transform(X_train_split)
    X_val_scaled = predictor.scaler.transform(X_val)
    X_test_scaled = predictor.scaler.transform(X_test)

    X_train_selected = predictor.feature_selector.fit_transform(X_train_scaled, y_train_split)
    X_val_selected = predictor.feature_selector.transform(X_val_scaled)
    X_test_selected = predictor.feature_selector.transform(X_test_scaled)

    # Handle class imbalance
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train_split)

    # Train multiple models
    predictor.train_multiple_models(X_train_balanced, y_train_balanced,
                                  X_val_selected, y_val, optimize=optimize_hyperparams)

    # Evaluate all models
    results = predictor.evaluate_models(X_test_selected, y_test)

    # Save enhanced model
    predictor.save_enhanced_model()

    return predictor, results

In [4]:
# ============================================================================
# BLOCK 4: ENHANCED INTERACTIVE PREDICTION
# ============================================================================

def enhanced_interactive_prediction():
    """Enhanced interactive prediction with all models"""
    print("="*60)
    print("ENHANCED INTERACTIVE TOXICITY PREDICTION")
    print("="*60)

    # Load enhanced model
    predictor = EnhancedToxicityPredictor()

    try:
        model_pkl_path = "Talksick-v2-ensemble_toxicity_model.pkl"
        nn_model_path = model_pkl_path.replace('.pkl', '_nn.h5') # Corrected filename

        with open(model_pkl_path, 'rb') as f:
            model_data = pickle.load(f)

        predictor.models = model_data['models']
        predictor.ensemble_model = model_data['ensemble_model']
        predictor.scaler = model_data['scaler']
        predictor.feature_selector = model_data['feature_selector']
        predictor.feature_names = model_data['feature_names']
        predictor.best_params = model_data['best_params']
        predictor.is_trained = True

        # Load neural network
        from tensorflow.keras.models import load_model
        predictor.nn_model = load_model(nn_model_path) # Corrected filename
        predictor.models['nn'] = predictor.nn_model # Add NN back to models dict after loading


        print("✅ Enhanced models loaded successfully")

    except Exception as e:
        print(f"❌ Error loading models: {e}")
        return

    def display_enhanced_results(smiles, predictions):
        """Display results from all models"""
        print("\n" + "="*60)
        print("ENHANCED TOXICITY PREDICTION RESULTS")
        print("="*60)
        print(f"SMILES: {smiles}")
        print("-"*60)

        for model_name, result in predictions.items():
            pred_text = 'Toxic' if result['prediction'] == 1 else 'Non-toxic'
            print(f"{model_name.upper()}: {pred_text} (prob: {result['probability']:.3f})")

        print("-"*60)

        # Consensus prediction
        toxic_votes = sum(1 for r in predictions.values() if r['prediction'] == 1)
        total_votes = len(predictions)
        consensus = "Toxic" if toxic_votes > total_votes/2 else "Non-toxic"
        confidence = max(toxic_votes, total_votes - toxic_votes) / total_votes

        print(f"CONSENSUS: {consensus} ({toxic_votes}/{total_votes} votes)")
        print(f"CONFIDENCE: {confidence:.1%}")
        print("="*60)

    # Main interaction loop
    while True:
        try:
            user_input = input("\nEnter SMILES string (or 'quit' to exit): ").strip()

            if user_input.lower() in ['quit', 'exit', 'q']:
                print("Goodbye!")
                break

            if not user_input:
                continue

            print("Processing with all models...")
            predictions, error = predictor.predict_single_molecule_enhanced(user_input)

            if error:
                print(f"❌ Error: {error}")
                continue

            display_enhanced_results(user_input, predictions)

        except KeyboardInterrupt:
            print("\n\nGoodbye!")
            break
        except Exception as e:
            print(f"❌ Unexpected error: {e}")

In [5]:
# ============================================================================
# QUICK START FUNCTIONS
# ============================================================================

def quick_train_enhanced(optimize=True):
    """Quick enhanced training"""
    return train_enhanced_toxicity_model(optimize_hyperparams=optimize)

def quick_predict_enhanced():
    """Quick enhanced prediction"""
    enhanced_interactive_prediction()

# Example usage:
# predictor, results = quick_train_enhanced(optimize=True)
# quick_predict_enhanced()

In [6]:
# BLOCK 5: MAIN EXECUTION FOR ENSEMBLE VERSION
# ============================================================================
import os

def main_ensemble():
    """Main function to run the ensemble toxicity prediction pipeline"""
    print("ENSEMBLE TOXICITY PREDICTION SYSTEM")
    print("=" * 60)

    model_pkl_path = "Talksick-v2-ensemble_toxicity_model.pkl"
    model_nn_path = model_pkl_path.replace('.pkl', '_nn.h5')

    # Check if model files exist
    model_exists = os.path.exists(model_pkl_path) and os.path.exists(model_nn_path)

    if model_exists:
        print("Trained model found. You can proceed with interactive prediction.")
        print("Choose an option:")
        print("1. Interactive prediction")
        print("2. Train a new model (overwrites existing)")
        choice = input("\nEnter your choice (1 or 2): ").strip()

        if choice == '1':
            quick_predict_enhanced()
        elif choice == '2':
            print("\n" + "=" * 60)
            print("Training a new model...")
            predictor, results = quick_train_enhanced(optimize=True)
            print("\n" + "=" * 60)
            print("Training complete! Starting interactive prediction with the new model...")
            quick_predict_enhanced()
        else:
            print("Invalid choice. Please run again and choose 1 or 2.")

    else:
        print("No trained model found. Training a new model...")
        predictor, results = quick_train_enhanced(optimize=True)
        print("\n" + "=" * 60)
        print("Training complete! Starting interactive prediction...")
        quick_predict_enhanced()

In [7]:
# ============================================================================
# BLOCK 6: FEATURE TOXICITY ANALYSIS
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from scipy.stats import chi2_contingency, pearsonr
import numpy as np
import pandas as pd
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

class ToxicityFeatureAnalyzer:
    """Comprehensive analysis of features contributing to molecular toxicity"""

    def __init__(self, predictor=None):
        """
        Initialize the analyzer with a trained predictor

        Args:
            predictor: Trained EnhancedToxicityPredictor instance
        """
        self.predictor = predictor
        self.feature_importance_results = {}
        self.statistical_results = {}
        self.correlation_results = {}

    def analyze_feature_importance(self, X_train, y_train, X_test, y_test, top_n=50):
        """
        Comprehensive feature importance analysis using multiple methods

        Args:
            X_train, y_train: Training data
            X_test, y_test: Test data
            top_n: Number of top features to analyze
        """
        print("🔍 Analyzing Feature Importance for Toxicity...")
        print("-" * 60)

        results = {}

        # 1. Random Forest Feature Importance
        print("1. Random Forest Feature Importance...")
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_train, y_train)
        rf_importance = rf.feature_importances_

        # 2. Permutation Importance (more reliable)
        print("2. Permutation Importance Analysis...")
        perm_importance = permutation_importance(
            rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
        )

        # 3. Logistic Regression Coefficients
        print("3. Logistic Regression Coefficients...")
        lr = LogisticRegression(random_state=42, max_iter=1000)
        lr.fit(X_train, y_train)
        lr_coefs = np.abs(lr.coef_[0])

        # Compile results
        feature_names = self.predictor.feature_names if self.predictor else [f"Feature_{i}" for i in range(len(rf_importance))]

        results_df = pd.DataFrame({
            'Feature': feature_names,
            'RF_Importance': rf_importance,
            'Perm_Importance_Mean': perm_importance.importances_mean,
            'Perm_Importance_Std': perm_importance.importances_std,
            'LR_Coefficient': lr_coefs
        })

        # Calculate composite importance score
        # Normalize each metric to 0-1 scale
        for col in ['RF_Importance', 'Perm_Importance_Mean', 'LR_Coefficient']:
            results_df[f'{col}_norm'] = (results_df[col] - results_df[col].min()) / (results_df[col].max() - results_df[col].min())

        results_df['Composite_Score'] = (
            results_df['RF_Importance_norm'] +
            results_df['Perm_Importance_Mean_norm'] +
            results_df['LR_Coefficient_norm']
        ) / 3

        # Sort by composite score
        results_df = results_df.sort_values('Composite_Score', ascending=False)

        self.feature_importance_results = results_df

        # Display top features
        print(f"\n🏆 TOP {top_n} MOST IMPORTANT FEATURES FOR TOXICITY:")
        print("=" * 80)
        print(f"{'Rank':<4} {'Feature':<40} {'Composite':<10} {'RF':<8} {'Perm':<8} {'LR':<8}")
        print("-" * 80)

        for i, (_, row) in enumerate(results_df.head(top_n).iterrows(), 1):
            print(f"{i:<4} {row['Feature'][:38]:<40} {row['Composite_Score']:.3f}     "
                  f"{row['RF_Importance']:.3f}   {row['Perm_Importance_Mean']:.3f}   {row['LR_Coefficient']:.3f}")

        return results_df

    def analyze_feature_statistics(self, X, y, top_features=None, top_n=20):
        """
        Statistical analysis of top features between toxic and non-toxic molecules

        Args:
            X: Feature matrix
            y: Target labels
            top_features: List of feature names to analyze (if None, uses top from importance)
            top_n: Number of top features to analyze
        """
        print(f"\n📊 Statistical Analysis of Top {top_n} Features...")
        print("-" * 60)

        if top_features is None and hasattr(self, 'feature_importance_results'):
            top_features = self.feature_importance_results.head(top_n)['Feature'].tolist()
        elif top_features is None:
            top_features = [f"Feature_{i}" for i in range(min(top_n, X.shape[1]))]

        feature_names = self.predictor.feature_names if self.predictor else [f"Feature_{i}" for i in range(X.shape[1])]
        X_df = pd.DataFrame(X, columns=feature_names)

        stats_results = []

        for feature in top_features:
            if feature not in X_df.columns:
                continue

            toxic_values = X_df[y == 1][feature]
            non_toxic_values = X_df[y == 0][feature]

            # Calculate statistics
            toxic_mean = toxic_values.mean()
            non_toxic_mean = non_toxic_values.mean()
            toxic_std = toxic_values.std()
            non_toxic_std = non_toxic_values.std()

            # Effect size (Cohen's d)
            pooled_std = np.sqrt(((len(toxic_values) - 1) * toxic_std**2 +
                                (len(non_toxic_values) - 1) * non_toxic_std**2) /
                               (len(toxic_values) + len(non_toxic_values) - 2))
            cohens_d = (toxic_mean - non_toxic_mean) / pooled_std if pooled_std > 0 else 0

            # Correlation with toxicity
            correlation, p_value = pearsonr(X_df[feature], y)

            stats_results.append({
                'Feature': feature,
                'Toxic_Mean': toxic_mean,
                'NonToxic_Mean': non_toxic_mean,
                'Difference': toxic_mean - non_toxic_mean,
                'Cohens_D': cohens_d,
                'Correlation': correlation,
                'P_Value': p_value,
                'Effect_Size': 'Large' if abs(cohens_d) > 0.8 else 'Medium' if abs(cohens_d) > 0.5 else 'Small'
            })

        stats_df = pd.DataFrame(stats_results)
        stats_df = stats_df.sort_values('Cohens_D', key=abs, ascending=False)

        self.statistical_results = stats_df

        # Display results
        cohens_d_header = "Cohen's D"
        print(f"\n{'Feature':<35} {'Toxic':<8} {'Non-Toxic':<10} {'Diff':<8} {cohens_d_header:<9} {'Corr':<6}")
        print("-" * 80)

        for _, row in stats_df.head(20).iterrows():
            print(f"{row['Feature'][:33]:<35} {row['Toxic_Mean']:.3f}    {row['NonToxic_Mean']:.3f}      "
                  f"{row['Difference']:.3f}   {row['Cohens_D']:.3f}     {row['Correlation']:.3f}")

        return stats_df

    def identify_toxicity_patterns(self, X, y, threshold_percentile=95):
        """
        Identify molecular patterns strongly associated with toxicity

        Args:
            X: Feature matrix
            y: Target labels
            threshold_percentile: Percentile threshold for high feature values
        """
        print(f"\n🔬 Identifying Toxicity Patterns (>{threshold_percentile}th percentile)...")
        print("-" * 60)

        feature_names = self.predictor.feature_names if self.predictor else [f"Feature_{i}" for i in range(X.shape[1])]
        X_df = pd.DataFrame(X, columns=feature_names)

        pattern_results = []

        for feature in feature_names:
            # Calculate threshold for "high" values
            threshold = np.percentile(X_df[feature], threshold_percentile)

            # Create binary indicator for high values
            high_feature = (X_df[feature] > threshold).astype(int)

            # Calculate toxicity rate for high vs low feature values
            if high_feature.sum() > 10:  # Only analyze if we have enough samples
                high_toxicity_rate = y[high_feature == 1].mean()
                low_toxicity_rate = y[high_feature == 0].mean()

                # Calculate odds ratio
                high_toxic = sum((high_feature == 1) & (y == 1))
                high_nontoxic = sum((high_feature == 1) & (y == 0))
                low_toxic = sum((high_feature == 0) & (y == 1))
                low_nontoxic = sum((high_feature == 0) & (y == 0))

                if high_nontoxic > 0 and low_nontoxic > 0:
                    odds_ratio = (high_toxic / high_nontoxic) / (low_toxic / low_nontoxic)
                else:
                    odds_ratio = float('inf') if high_nontoxic == 0 else 0

                pattern_results.append({
                    'Feature': feature,
                    'Threshold': threshold,
                    'High_Samples': high_feature.sum(),
                    'High_Toxicity_Rate': high_toxicity_rate,
                    'Low_Toxicity_Rate': low_toxicity_rate,
                    'Rate_Difference': high_toxicity_rate - low_toxicity_rate,
                    'Odds_Ratio': odds_ratio
                })

        pattern_df = pd.DataFrame(pattern_results)
        pattern_df = pattern_df.sort_values('Rate_Difference', ascending=False)

        # Display top toxicity-indicating patterns
        print(f"\n🚨 TOP TOXICITY-INDICATING PATTERNS:")
        print("=" * 85)
        print(f"{'Feature':<35} {'High Tox%':<9} {'Low Tox%':<8} {'Diff':<6} {'Odds Ratio':<10}")
        print("-" * 85)

        for _, row in pattern_df.head(15).iterrows():
            if row['Rate_Difference'] > 0:  # Only show features that increase toxicity
                print(f"{row['Feature'][:33]:<35} {row['High_Toxicity_Rate']:.1%}       "
                      f"{row['Low_Toxicity_Rate']:.1%}      {row['Rate_Difference']:.3f}  {row['Odds_Ratio']:.2f}")

        return pattern_df

    def create_toxicity_report(self, output_file="toxicity_feature_report.txt"):
        """Generate a comprehensive toxicity feature analysis report"""

        print(f"\n📝 Generating Comprehensive Toxicity Report...")
        print(f"Report will be saved to: {output_file}")

        with open(output_file, 'w') as f:
            f.write("=" * 80 + "\n")
            f.write("MOLECULAR TOXICITY FEATURE ANALYSIS REPORT\n")
            f.write("=" * 80 + "\n\n")

            # Summary statistics
            if hasattr(self, 'statistical_results') and not self.statistical_results.empty:
                f.write("EXECUTIVE SUMMARY\n")
                f.write("-" * 40 + "\n")

                high_impact_features = self.statistical_results[abs(self.statistical_results['Cohens_D']) > 0.5]
                f.write(f"• {len(high_impact_features)} features show medium-to-large effect sizes\n")

                strong_correlations = self.statistical_results[abs(self.statistical_results['Correlation']) > 0.3]
                f.write(f"• {len(strong_correlations)} features show strong correlation with toxicity\n")

                f.write(f"• Most toxic-indicating feature: {self.statistical_results.iloc[0]['Feature']}\n")
                cohens_d_val = self.statistical_results.iloc[0]['Cohens_D']
                f.write(f"  (Cohen's D: {cohens_d_val:.3f})\n\n")

            # Top important features
            if hasattr(self, 'feature_importance_results') and not self.feature_importance_results.empty:
                f.write("TOP 20 MOST IMPORTANT FEATURES FOR TOXICITY PREDICTION\n")
                f.write("-" * 60 + "\n")
                f.write(f"{'Rank':<4} {'Feature':<40} {'Composite Score':<15}\n")
                f.write("-" * 60 + "\n")

                for i, (_, row) in enumerate(self.feature_importance_results.head(20).iterrows(), 1):
                    f.write(f"{i:<4} {row['Feature'][:38]:<40} {row['Composite_Score']:.4f}\n")
                f.write("\n")

            # Statistical analysis
            if hasattr(self, 'statistical_results') and not self.statistical_results.empty:
                f.write("STATISTICAL DIFFERENCES BETWEEN TOXIC AND NON-TOXIC MOLECULES\n")
                f.write("-" * 65 + "\n")
                f.write(f"{'Feature':<35} {'Effect Size':<10} {'Correlation':<12}\n")
                f.write("-" * 65 + "\n")

                for _, row in self.statistical_results.head(20).iterrows():
                    f.write(f"{row['Feature'][:33]:<35} {row['Effect_Size']:<10} {row['Correlation']:.3f}\n")
                f.write("\n")

            f.write("INTERPRETATION GUIDE\n")
            f.write("-" * 20 + "\n")
            f.write("• Composite Score: Combined importance from multiple ML algorithms\n")
            cohens_d_guide = "• Cohen's D: Effect size (>0.8=Large, >0.5=Medium, >0.2=Small)\n"
            f.write(cohens_d_guide)
            f.write("• Correlation: Linear relationship with toxicity (-1 to +1)\n")
            f.write("• Positive values indicate features associated with toxicity\n")
            f.write("• Negative values indicate features associated with safety\n\n")

            f.write("=" * 80 + "\n")
            f.write("Report generated by Enhanced Toxicity Prediction System\n")
            f.write("=" * 80 + "\n")

    def visualize_top_features(self, top_n=20, save_plots=True):
        """Create visualizations of top toxicity-related features"""

        if not hasattr(self, 'feature_importance_results') or self.feature_importance_results.empty:
            print("❌ No feature importance results found. Run analyze_feature_importance first.")
            return

        print(f"📊 Creating visualizations for top {top_n} features...")

        # Create figure with subplots
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))

        top_features = self.feature_importance_results.head(top_n)

        # 1. Feature Importance Comparison
        ax1 = axes[0, 0]
        x_pos = np.arange(min(10, len(top_features)))

        ax1.barh(x_pos, top_features.head(10)['Composite_Score'], alpha=0.7, color='skyblue')
        ax1.set_yticks(x_pos)
        ax1.set_yticklabels([f[:25] + '...' if len(f) > 25 else f for f in top_features.head(10)['Feature']])
        ax1.set_xlabel('Composite Importance Score')
        ax1.set_title('Top 10 Most Important Features')
        ax1.grid(axis='x', alpha=0.3)

        # 2. Method Comparison
        ax2 = axes[0, 1]
        methods = ['RF_Importance', 'Perm_Importance_Mean', 'LR_Coefficient']
        method_labels = ['Random Forest', 'Permutation', 'Logistic Regression']

        for i, (method, label) in enumerate(zip(methods, method_labels)):
            ax2.scatter(top_features.head(10)[method], range(10),
                       alpha=0.7, label=label, s=50)

        ax2.set_ylabel('Feature Rank')
        ax2.set_xlabel('Importance Score')
        ax2.set_title('Importance by Different Methods')
        ax2.legend()
        ax2.grid(alpha=0.3)

        # 3. Statistical Analysis (if available)
        if hasattr(self, 'statistical_results') and not self.statistical_results.empty:
            ax3 = axes[1, 0]
            stats_top = self.statistical_results.head(10)

            colors = ['red' if d > 0 else 'blue' for d in stats_top['Cohens_D']]
            ax3.barh(range(10), stats_top['Cohens_D'], color=colors, alpha=0.7)
            ax3.set_yticks(range(10))
            ax3.set_yticklabels([f[:20] + '...' if len(f) > 20 else f for f in stats_top['Feature']])
            ax3.set_xlabel("Cohen's D (Effect Size)")
            effect_size_title = "Effect Sizes (Red=Pro-toxic, Blue=Protective)"
            ax3.set_title(effect_size_title)
            ax3.axvline(x=0, color='black', linestyle='-', alpha=0.3)
            ax3.grid(axis='x', alpha=0.3)

        # 4. Correlation plot (if available)
        if hasattr(self, 'statistical_results') and not self.statistical_results.empty:
            ax4 = axes[1, 1]

            ax4.scatter(self.statistical_results['Correlation'],
                       abs(self.statistical_results['Cohens_D']),
                       alpha=0.6, s=50, c='green')
            ax4.set_xlabel('Correlation with Toxicity')
            ax4.set_ylabel('Absolute Effect Size')
            ax4.set_title('Correlation vs Effect Size')
            ax4.grid(alpha=0.3)

            # Add quadrant labels
            ax4.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
            ax4.axvline(x=0, color='red', linestyle='--', alpha=0.5)

        plt.tight_layout()

        if save_plots:
            plt.savefig('toxicity_feature_analysis.png', dpi=300, bbox_inches='tight')
            print("✅ Plots saved as 'toxicity_feature_analysis.png'")

        plt.show()

def analyze_toxicity_features_with_existing_predictor(predictor, X_data, y_data):
    """
    Analyze toxicity features using an existing trained predictor and processed data

    Args:
        predictor: Your trained EnhancedToxicityPredictor instance
        X_data: Preprocessed feature matrix (already scaled and selected)
        y_data: Target labels
    """
    print("🧬 COMPREHENSIVE TOXICITY FEATURE ANALYSIS")
    print("=" * 60)

    # Initialize analyzer
    analyzer = ToxicityFeatureAnalyzer(predictor)

    # Split data for analysis (same as training)
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
    )

    print(f"Data shapes: X_train={X_train.shape}, X_test={X_test.shape}")
    print(f"Class distribution: Toxic={sum(y_data)}, Non-toxic={len(y_data)-sum(y_data)}")

    # Perform comprehensive analysis
    print("\n🔍 Starting comprehensive feature analysis...")

    try:
        # 1. Feature Importance Analysis
        print("Step 1: Feature Importance Analysis...")
        importance_results = analyzer.analyze_feature_importance(X_train, y_train, X_test, y_test)

        # 2. Statistical Analysis
        print("Step 2: Statistical Analysis...")
        stats_results = analyzer.analyze_feature_statistics(X_data, y_data)

        # 3. Toxicity Pattern Analysis
        print("Step 3: Toxicity Pattern Analysis...")
        pattern_results = analyzer.identify_toxicity_patterns(X_data, y_data)

        # 4. Generate Report
        print("Step 4: Generating Report...")
        analyzer.create_toxicity_report()

        # 5. Create Visualizations
        print("Step 5: Creating Visualizations...")
        analyzer.visualize_top_features()

        print("\n✅ Analysis Complete!")
        print("📊 Results saved:")
        print("   • toxicity_feature_report.txt")
        print("   • toxicity_feature_analysis.png")

    except Exception as e:
        print(f"❌ Error during analysis: {e}")
        import traceback
        traceback.print_exc()
        return None

    return analyzer

def analyze_toxicity_features(input_file="sr-mmp-monol.smiles",
                             model_pkl_path="Talksick-v2-ensemble_toxicity_model.pkl"):
    """
    Main function to perform comprehensive toxicity feature analysis
    This version tries to work with the saved model data directly
    """
    print("🧬 COMPREHENSIVE TOXICITY FEATURE ANALYSIS")
    print("=" * 60)

    # Load the trained model
    try:
        import pickle
        with open(model_pkl_path, 'rb') as f:
            model_data = pickle.load(f)

        print("✅ Trained model loaded successfully")
        print(f"Available keys in model: {list(model_data.keys())}")

        # Create a mock predictor object with necessary attributes
        class MockPredictor:
            def __init__(self, model_data):
                self.models = model_data.get('models', {})
                self.scaler = model_data.get('scaler', None)
                self.feature_selector = model_data.get('feature_selector', None)
                self.feature_names = model_data.get('feature_names', [])
                self.is_trained = True
                print(f"Number of features: {len(self.feature_names)}")

        predictor = MockPredictor(model_data)

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        import traceback
        traceback.print_exc()
        return None

    # Load and prepare data - simplified approach
    try:
        print(f"Loading data from {input_file}...")
        df = pd.read_csv(input_file, sep="\s+", header=None, usecols=[0, 2], names=["SMILES", "MMP"])
        print(f"Loaded {len(df)} molecules")

        # For this analysis, we'll create dummy feature data that matches the saved model's expectations
        # In reality, you'd need to recompute features using your actual feature computation pipeline

        print("⚠️  WARNING: Using dummy feature data for demonstration.")
        print("   For real analysis, integrate this with your actual feature computation pipeline.")

        # Create dummy features matching the expected shape
        n_features = len(predictor.feature_names) if predictor.feature_names else 100
        n_samples = len(df)

        # Generate some realistic dummy features
        np.random.seed(42)
        X_dummy = np.random.randn(n_samples, n_features)
        y_dummy = df['MMP'].values

        print(f"Using dummy feature matrix shape: {X_dummy.shape}")
        print(f"Target distribution: {np.bincount(y_dummy)}")

        return analyze_toxicity_features_with_existing_predictor(predictor, X_dummy, y_dummy)

    except Exception as e:
        print(f"❌ Error processing data: {e}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# EASY INTEGRATION WITH YOUR EXISTING CODE
# ============================================================================

def analyze_features_from_training_data(predictor, df_features):
    """
    RECOMMENDED: Use this function if you have access to your training pipeline

    Args:
        predictor: Your trained EnhancedToxicityPredictor instance
        df_features: DataFrame with computed features (output from compute_features)
    """
    print("🧬 ANALYZING FEATURES FROM TRAINING DATA")
    print("=" * 60)

    # Prepare features exactly as during training
    X = df_features.drop(columns=['MMP', 'SMILES']).values
    y = df_features['MMP'].values

    # Apply same preprocessing
    X_scaled = predictor.scaler.transform(X)
    X_selected = predictor.feature_selector.transform(X_scaled)

    print(f"Feature matrix shape: {X_selected.shape}")
    print(f"Number of features: {len(predictor.feature_names)}")

    return analyze_toxicity_features_with_existing_predictor(predictor, X_selected, y)

# ============================================================================
# QUICK START FUNCTIONS (Updated)
# ============================================================================

def quick_analyze_toxicity_features():
    """
    Quick function to analyze toxicity features
    NOTE: This uses dummy data - integrate with your actual pipeline for real results
    """
    print("⚠️  Using dummy data for demonstration.")
    print("   For real analysis, use analyze_features_from_training_data() instead.")
    return analyze_toxicity_features()

def quick_analyze_with_real_data(predictor, df_features):
    """
    Quick function to analyze with your real training data

    Usage:
        # After training your model:
        predictor, results = train_enhanced_toxicity_model()

        # Load and process data (same as training):
        df = pd.read_csv("sr-mmp-monol.smiles", sep="\s+", header=None, usecols=[0, 2], names=["SMILES", "MMP"])
        df = predictor.validate_smiles(df)
        df_features = predictor.compute_features(df)

        # Analyze features:
        analyzer = quick_analyze_with_real_data(predictor, df_features)
    """
    return analyze_features_from_training_data(predictor, df_features)

# Example usage:
# analyzer = quick_analyze_toxicity_features()

In [8]:
# ============================================================================
# BLOCK 7: FEATURE ANALYSIS EXECUTION (ADD THIS NEW BLOCK)
# ============================================================================

def run_feature_analysis():
    """Run comprehensive feature analysis on trained model"""
    print("🧬 STARTING FEATURE ANALYSIS")
    print("=" * 60)

    # After training your model
    predictor, results = train_enhanced_toxicity_model()

    # Load and process data (same as your training process)
    df = pd.read_csv("sr-mmp-monol.smiles", sep="\s+", header=None, usecols=[0, 2], names=["SMILES", "MMP"])
    df = predictor.validate_smiles(df)
    df_features = predictor.compute_features(df)

    # Now analyze the features
    analyzer = quick_analyze_with_real_data(predictor, df_features)

    return analyzer

def main_ensemble_with_analysis():
    """Main function with feature analysis option"""
    print("ENSEMBLE TOXICITY PREDICTION SYSTEM")
    print("=" * 60)

    model_pkl_path = "Talksick-v2-ensemble_toxicity_model.pkl"
    model_nn_path = model_pkl_path.replace('.pkl', '_nn.h5')
    model_exists = os.path.exists(model_pkl_path) and os.path.exists(model_nn_path)

    if model_exists:
        print("Trained model found. Choose an option:")
        print("1. Interactive prediction")
        print("2. Train a new model")
        print("3. Feature analysis (analyze features contributing to MMP disruption)")
        choice = input("\nEnter your choice (1, 2, or 3): ").strip()

        if choice == '1':
            quick_predict_enhanced()
        elif choice == '2':
            print("Training a new model...")
            predictor, results = quick_train_enhanced(optimize=True)
            print("Training complete! Starting interactive prediction...")
            quick_predict_enhanced()
        elif choice == '3':
            print("Running feature analysis...")
            analyzer = run_feature_analysis()
        else:
            print("Invalid choice. Please run again and choose 1, 2, or 3.")

    else:
        print("No trained model found. Choose an option:")
        print("1. Train new model")
        print("2. Train new model + run feature analysis")
        choice = input("\nEnter your choice (1 or 2): ").strip()

        if choice == '1':
            predictor, results = quick_train_enhanced(optimize=True)
            print("Training complete! Starting interactive prediction...")
            quick_predict_enhanced()
        elif choice == '2':
            print("Training model and running feature analysis...")
            analyzer = run_feature_analysis()
        else:
            print("Invalid choice. Please run again and choose 1 or 2.")

In [9]:
# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    main_ensemble_with_analysis()  # Changed from main_ensemble()



ENSEMBLE TOXICITY PREDICTION SYSTEM
Trained model found. Choose an option:
1. Interactive prediction
2. Train a new model
3. Feature analysis (analyze features contributing to MMP disruption)

Enter your choice (1, 2, or 3): 1
ENHANCED INTERACTIVE TOXICITY PREDICTION


✅ Enhanced models loaded successfully

Enter SMILES string (or 'quit' to exit): C1=CC(=CC=C1NN=C(C#N)C#N)OC(F)(F)F
Processing with all models...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step

ENHANCED TOXICITY PREDICTION RESULTS
SMILES: C1=CC(=CC=C1NN=C(C#N)C#N)OC(F)(F)F
------------------------------------------------------------
RF: Toxic (prob: 0.647)
XGB: Toxic (prob: 0.974)
SVM: Toxic (prob: 1.000)
LGB: Toxic (prob: 0.922)
NN: Toxic (prob: 0.962)
ENSEMBLE: Toxic (prob: 0.886)
------------------------------------------------------------
CONSENSUS: Toxic (6/6 votes)
CONFIDENCE: 100.0%


Exception ignored in: <function Booster.__del__ at 0x7f55b7c2f060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/xgboost/core.py", line 1932, in __del__
    def __del__(self) -> None:

KeyboardInterrupt: 




Goodbye!
